# SHAP-Based Patient-Level Explanation of Mortality Risk Predictions

In [5]:
import joblib
import pandas as np
from sklearn.pipeline import Pipeline

# Load Saved Models and Data

In [7]:
# We load our best models
best_logistic_model = joblib.load("../machine learning modeling/best_logistic_model.pkl")
best_model_gb = joblib.load("../machine learning modeling/best_gradient_boosting_model.pkl")
X_train = joblib.load("../machine learning modeling/X_train.pkl")
X_test = joblib.load("../machine learning modeling/X_test.pkl")
y_train = joblib.load("../machine learning modeling/y_train.pkl")
y_test = joblib.load("../machine learning modeling/y_test.pkl")

In [30]:
print(type(best_logistic_model))
print(type(best_model_gb))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.pipeline.Pipeline'>


This shows the saved models are pipeline. So, we double check the pipeline steps.

In [10]:
if isinstance(best_logistic_model, Pipeline):
    print("Logistic Pipeline Steps:", best_logistic_model.named_steps.keys())

if isinstance(best_model_gb, Pipeline):
    print("Gradient Boosting Pipeline Steps:", best_model_gb.named_steps.keys())

Logistic Pipeline Steps: dict_keys(['preprocessor', 'classifier'])
Gradient Boosting Pipeline Steps: dict_keys(['preprocessor', 'classifier'])


We separate the preprocessing step from the estimator (classifier) so SHAP explainer sees the transformed features that the classifier uses.

In [12]:
def split_pipeline(model):
    if isinstance(model, Pipeline):
        preprocessor = Pipeline(model.steps[:-1]) # returns everything but the last item in the pipeline
        estimator = model.steps[-1][1]
        return preprocessor, estimator

    return None, model

log_preprocess, log_estimator = split_pipeline(best_logistic_model)
gb_preprocess, gb_estimator = split_pipeline(best_model_gb)

In [14]:
print(log_estimator)

LogisticRegression(C=0.03, class_weight='balanced', max_iter=1000, penalty='l2',
                   random_state=42)


In [15]:
print(gb_estimator)

HistGradientBoostingClassifier(class_weight='balanced', early_stopping=True,
                               l2_regularization=np.float64(1.025616274847307),
                               learning_rate=np.float64(0.012502377950801122),
                               max_features=0.6, max_iter=847,
                               max_leaf_nodes=77, n_iter_no_change=20,
                               random_state=42, validation_fraction=0.15)


# Prepare Data for SHAP

We start by defining functions that will help us transform our data

In [34]:
# After preprocessing, the transformed matrix may be a sparse matrix,
# which is not preferred for ploting, so we convert to dense matrix

def to_dense(X):
    if hasattr(X, "toarray"):        #sparse matrix has method "toarray" attributes
        return X.to_array()         # converts to dense arrays
    return X
        

def get_feature_names(preprocess, X_raw):
    """ We will try to extract the feature names and clean it, and if it fails we resort to default names"""
    try:
        names = preprocess.get_feature_names_out()
    except Exception:
        names = [f"feature_{i}" for i in range(preprocess.transform(X_raw).shape[1])]

    cleaned = []
    for name in names:
        cleaned_name = str(name)
        cleaned_name = cleaned_name.replace("num__", "")
        cleaned_name = cleaned_name.replace("cat__", "")
        cleaned.append(cleaned_name)
    return cleaned

print(get_feature_names(log_preprocess, X_train))

['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium', 'anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']
